In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler

# Load Data in Datafreme

In [2]:
df = pd.read_csv('titanic_data_updated.csv')
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,no,second,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,yes,first,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,no,third,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,yes,first,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


# Splitting Data

In [3]:
df.drop(['PassengerId', 'Name', 'Ticket'], axis=1, inplace=True)

# Family_size creation
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# Feature and Target extract
X = df.drop(['Survived'], axis=1)
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Imputation of Data

In [4]:
imputer_transformer = ColumnTransformer(
    transformers = [
        ('age', SimpleImputer(missing_values=np.nan, strategy='mean'), ['Age']),
        ('embarked', SimpleImputer(missing_values=np.nan, strategy='most_frequent'), ['Embarked']),
        ('cavin', SimpleImputer(missing_values=np.nan, strategy='constant', fill_value='Missing', add_indicator=True), ['Cabin'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

imputer_transformer.set_output(transform='pandas')

imputer_transformer.fit(X_train)
X_train = imputer_transformer.transform(X_train)
X_test = imputer_transformer.transform(X_test)

In [5]:
print(f"-----------X_train------------")
print(X_train.isnull().sum())
print(f"\n-----------X_test------------")
print(X_test.isnull().sum())

-----------X_train------------
Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
SibSp                     0
Parch                     0
Fare                      0
FamilySize                0
dtype: int64

-----------X_test------------
Age                       0
Embarked                  0
Cabin                     0
missingindicator_Cabin    0
Pclass                    0
Sex                       0
SibSp                     0
Parch                     0
Fare                      0
FamilySize                0
dtype: int64


# Outlier Handling

In [6]:
# outlier handling of age

mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()
X_train['zscore_age'] = (X_train['Age'] - mean_age) / std_age

X_train = X_train[abs(X_train['zscore_age']) <= 3]
X_train.drop(['zscore_age'], axis=1, inplace=True)

In [7]:
# outlier handling of fare

fare_Q1 = X_train['Fare'].quantile(0.25)
fare_Q3 = X_train['Fare'].quantile(0.75)
fare_IQR = fare_Q1 - fare_Q3

fare_min = max(0, fare_Q1-1.5 * fare_IQR)
fare_max = (fare_Q3+1.5 * fare_IQR)

X_train['Fare'] = X_train['Fare'].clip(fare_min, fare_max)

# Encoding & Scaling

In [12]:
X_train['Cabin_deck'] = X_train['Cabin'].astype(str).str[0]
X_test['Cabin_deck'] = X_test['Cabin'].astype(str).str[0]

In [14]:
encoder_scaler = ColumnTransformer(
    transformers=[
        ('pclass', OrdinalEncoder(categories=[['third', 'second', 'first']]), ['Pclass']),
        ('embarked_sex_deck', OneHotEncoder(sparse_output=False), ['Embarked', 'Sex', 'Cabin_deck']),

        ('age', StandardScaler(), ['Age']),
        ('fare_family', MinMaxScaler(), ['Fare', 'FamilySize'])
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

encoder_scaler.set_output(transform='pandas'),

encoder_scaler.fit(X_train)
X_train = encoder_scaler.transform(X_train)
X_test = encoder_scaler.transform(X_test)

In [16]:
X_train.drop(['Cabin', 'SibSp', 'Parch'], axis=1, inplace=True)
X_test.drop(['Cabin', 'SibSp', 'Parch'], axis=1, inplace=True)

# Final Dataset

In [17]:
X_train

,Pclass,Embarked_C,Embarked_Q,Embarked_S,Sex_female,Sex_male,Cabin_deck_A,Cabin_deck_B,Cabin_deck_C,Cabin_deck_D,Cabin_deck_E,Cabin_deck_F,Cabin_deck_G,Cabin_deck_M,Cabin_deck_T,Age,Fare,FamilySize,missingindicator_Cabin,cabin_dake
331,2.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.304495,0.682022,0.0,False,C
733,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.495295,0.311098,0.0,True,M
382,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.224621,0.189650,0.0,True,M
704,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.255323,0.187956,0.1,True,M
813,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.855135,0.748430,0.6,True,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
106,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.655276,0.183069,0.0,True,M
270,2.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.741849,0.0,True,M
860,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.944537,0.337620,0.2,True,M
435,2.0,0.0,0.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-1.215210,1.000000,0.3,False,B


In [18]:
X_test

,Pclass,Embarked_C,Embarked_Q,Embarked_S,Sex_female,Sex_male,Cabin_deck_A,Cabin_deck_B,Cabin_deck_C,Cabin_deck_D,Cabin_deck_E,Cabin_deck_F,Cabin_deck_G,Cabin_deck_M,Cabin_deck_T,Age,Fare,FamilySize,missingindicator_Cabin,cabin_dake
709,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.364841,0.2,True,M
439,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.144630,0.251271,0.0,True,M
840,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.735267,0.189650,0.0,True,M
720,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.855135,0.789710,0.1,True,M
39,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-1.215210,0.269021,0.1,True,M
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
433,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.975238,0.170506,0.0,True,M
773,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.024552,0.172899,0.0,True,M
25,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.704565,0.751122,0.6,True,M
84,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.975238,0.251271,0.0,True,M
